In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv("../data/togo-dapaong_qc.csv")
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Loaded 525600 rows, 19 columns


,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,WD,WDstdev,BP,Cleaning,Precipitation,TModA,TModB,Comments
0,2021-10-25 00:01,-1.3,0.0,0.0,0.0,0.0,24.8,94.5,0.9,1.1,0.4,227.6,1.1,977,0,0.0,24.7,24.4,NaN
1,2021-10-25 00:02,-1.3,0.0,0.0,0.0,0.0,24.8,94.4,1.1,1.6,0.4,229.3,0.7,977,0,0.0,24.7,24.4,NaN
2,2021-10-25 00:03,-1.3,0.0,0.0,0.0,0.0,24.8,94.4,1.2,1.4,0.3,228.5,2.9,977,0,0.0,24.7,24.4,NaN
3,2021-10-25 00:04,-1.2,0.0,0.0,0.0,0.0,24.8,94.3,1.2,1.6,0.3,229.1,4.6,977,0,0.0,24.7,24.4,NaN
4,2021-10-25 00:05,-1.2,0.0,0.0,0.0,0.0,24.8,94.0,1.3,1.6,0.4,227.5,1.6,977,0,0.0,24.7,24.4,NaN


In [3]:
print("=== Summary Statistics ===")
print(df.describe())

print("\n=== Missing Values ===")
missing = df.isna().sum()
print(missing[missing > 0])

# Check for >5% missing
pct_missing = (missing / len(df)) * 100
high_missing = pct_missing[pct_missing > 5]
if not high_missing.empty:
    print("\n>5% missing in:")
    print(high_missing)
else:
    print("\n✅ No columns with >5% missing.")

=== Summary Statistics ===
                 GHI            DNI            DHI           ModA  \
count  525600.000000  525600.000000  525600.000000  525600.000000   
mean      230.555040     151.258469     116.444352     226.144375   
std       322.532347     250.956962     156.520714     317.346938   
min       -12.700000       0.000000       0.000000       0.000000   
25%        -2.200000       0.000000       0.000000       0.000000   
50%         2.100000       0.000000       2.500000       4.400000   
75%       442.400000     246.400000     215.700000     422.525000   
max      1424.000000    1004.500000     805.700000    1380.000000   

                ModB           Tamb             RH             WS  \
count  525600.000000  525600.000000  525600.000000  525600.000000   
mean      219.568588      27.751788      55.013160       2.368093   
std       307.932510       4.758023      28.778732       1.462668   
min         0.000000      14.900000       3.300000       0.000000   
25%   

In [4]:
key_cols = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'WS', 'WSgust']
valid_key_cols = [c for c in key_cols if c in df.columns]

if valid_key_cols:
    z_scores = np.abs(stats.zscore(df[valid_key_cols].dropna()))
    outliers = (z_scores > 3).any(axis=1)
    print(f"Rows with |Z| > 3: {outliers.sum()}")
    df['is_outlier'] = False
    df.loc[df[valid_key_cols].dropna().index[outliers], 'is_outlier'] = True
else:
    print("⚠️ Key columns not found. Available:", df.columns.tolist())

Rows with |Z| > 3: 9251


In [5]:
# Impute missing values with median
for col in valid_key_cols:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# Fix timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df.sort_values('Timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_17268\3790170879.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_17268\3790170879.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, wh

In [7]:
df.to_csv("../data/togo_clean.csv", index=False)
print("✅ Cleaned Togo data saved.")

✅ Cleaned Togo data saved.


## 🔍 Key Insights — Togo dapaong

- GHI behavior shows clear diurnal pattern.
- ModA/ModB correlation indicates stable sensor performance.
- Outliers likely due to sensor noise or calibration drift.
- Wind speed shows moderate inverse relationship with GHI.

## 📚 References
- [EDA Best Practices](https://towardsdatascience.com/exploratory-data-analysis-eda-a-practical-guide...)
- SciPy Z-score docs
- Seaborn visualization guide